<a href="https://colab.research.google.com/github/BonnieLin1123/Cyberattack-Detection-ml/blob/main/Cyberattack-Detection-ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd

# The NSL-KDD files have no header row, so you add the column names manually
col_names = [
    "duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
    "wrong_fragment","urgent","hot","num_failed_logins","logged_in","num_compromised",
    "root_shell","su_attempted","num_root","num_file_creations","num_shells",
    "num_access_files","num_outbound_cmds","is_host_login","is_guest_login","count",
    "srv_count","serror_rate","srv_serror_rate","rerror_rate","srv_rerror_rate",
    "same_srv_rate","diff_srv_rate","srv_diff_host_rate","dst_host_count",
    "dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate",
    "dst_host_srv_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate",
    "label","difficulty"
]

train_df = pd.read_csv("KDDTrain+.txt", names=col_names)

# See how many examples of each attack type you have
print(train_df['label'].value_counts())

label
normal             67343
neptune            41214
satan               3633
ipsweep             3599
portsweep           2931
smurf               2646
nmap                1493
back                 956
teardrop             892
warezclient          890
pod                  201
guess_passwd          53
buffer_overflow       30
warezmaster           20
land                  18
imap                  11
rootkit               10
loadmodule             9
ftp_write              8
multihop               7
phf                    4
perl                   3
spy                    2
Name: count, dtype: int64


In [3]:
attack_map = {
    'normal': 'Normal',
    # DoS attacks
    'back':'DoS','land':'DoS','neptune':'DoS','pod':'DoS','smurf':'DoS','teardrop':'DoS',
    # Probe attacks
    'ipsweep':'Probe','nmap':'Probe','portsweep':'Probe','satan':'Probe',
    # R2L attacks
    'ftp_write':'R2L','guess_passwd':'R2L','imap':'R2L','multihop':'R2L',
    'phf':'R2L','spy':'R2L','warezclient':'R2L','warezmaster':'R2L',
    # U2R attacks
    'buffer_overflow':'U2R','loadmodule':'U2R','perl':'U2R','rootkit':'U2R'
}

train_df['attack_class'] = train_df['label'].map(attack_map)
print(train_df['attack_class'].value_counts())

attack_class
Normal    67343
DoS       45927
Probe     11656
R2L         995
U2R          52
Name: count, dtype: int64


In [4]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# The 3 categorical columns need to be encoded as numbers first
from sklearn.preprocessing import LabelEncoder
for col in ['protocol_type', 'service', 'flag']:
    train_df[col] = LabelEncoder().fit_transform(train_df[col])

# Take only the 41 feature columns (not label/difficulty)
feature_cols = col_names[:41]
X = train_df[feature_cols].values

# Normalize everything to 0–1 range (like pixel values)
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Reshape 41 features → pad to 49 → 7×7 grid
X_padded = np.pad(X_scaled, ((0,0),(0,8)), constant_values=0)  # pad to 49
X_grid = X_padded.reshape(-1, 7, 7)  # shape: (N, 7, 7)

# Resize to 32×32 (what ResNet-18 expects)
import torch
import torch.nn.functional as F
X_tensor = torch.tensor(X_grid, dtype=torch.float32).unsqueeze(1)  # add channel dim
X_resized = F.interpolate(X_tensor, size=(32, 32), mode='bilinear', align_corners=False)
# Final shape: (N, 1, 32, 32) — one grayscale channel